In [1]:
from PIL import Image
import os
import numpy as np
import torch
import pandas as pd
from math import ceil
from tqdm import tqdm
from huggingface_hub import login

metadata_path = "/tcga/open-access/gdc_data_portal/biospecimen/tcga_Biospecimen_SAMPLE_METADATA/2023-09-01/gdc_sample_sheet.2023-09-05.tsv"
metadata_df = pd.read_csv(metadata_path, sep='\t')
slides_df = metadata_df[metadata_df['Data Type'] == 'Slide Image']
slides_df = slides_df.sort_values(by='Project ID').reset_index(drop=True)
base_dir = '/tcga/open-access/gdc_data_portal/biospecimen/tcga_Biospecimen_FILES/'
slides_df['Full Path'] = slides_df.apply(lambda row: os.path.join(base_dir, row['File ID'], row['File Name']), axis=1)

num_slides = 250
num_patches_per_slide = 250
patch_size = 224

brca_embeddings = []
luad_embeddings = []
lusc_embeddings = []
coad_embeddings = []

In [2]:
pip install git+https://github.com/Mahmoodlab/CONCH.git

  Cloning https://github.com/Mahmoodlab/CONCH.git to /tmp/pip-req-build-9scrthrb
  Running command git clone --filter=blob:none --quiet https://github.com/Mahmoodlab/CONCH.git /tmp/pip-req-build-9scrthrb
  Resolved https://github.com/Mahmoodlab/CONCH.git to commit 171f2be94d8871fa9af72de6b86685c135333ee8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [8]:
# Run this: !rm conch.py if not working

In [3]:
import os
import numpy as np
import torch
from PIL import Image
from conch.open_clip_custom import create_model_from_pretrained
import matplotlib.pyplot as plt
import seaborn as sns
from rsatoolbox.data import Dataset
from rsatoolbox.rdm import calc_rdm

/homes2/vmishra/miniconda3/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [4]:
model, preprocess = create_model_from_pretrained(
    'conch_ViT-B-16',
    "hf_hub:MahmoodLab/conch",
    hf_auth_token="YOUR_TOKEN"
)
model.eval()
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
model.to(device)

/homes2/vmishra/miniconda3/lib/python3.12/site-packages/conch/open_clip_custom/factory.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoin

CoCa(
  (text): TextTransformer(
    (token_embedding): Embedding(32007, 768)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_final): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (visual): VisualModel(
    (trunk): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kerne

In [5]:
preprocessed_patches_dir_brca = "preprocessed_patches_BRCA"
preprocessed_patches_dir_luad = "/lotterlab/users/vmishra/preprocessed_patches_LUAD"
preprocessed_patches_dir_lusc = "/lotterlab/users/vmishra/preprocessed_patches_LUSC"
preprocessed_patches_dir_coad = "/lotterlab/users/vmishra/preprocessed_patches_COAD"

def load_patches_brca(category_label):
    patches_list = []
    filenames = [f for f in os.listdir(preprocessed_patches_dir_brca) if category_label in f]
    for filename in filenames:
        patches = np.load(os.path.join(preprocessed_patches_dir_brca, filename))
        patches_list.append(patches)
    return np.concatenate(patches_list, axis=0) if patches_list else np.array([])


def load_patches_luad(category_label):
    patches_list = []
    filenames = [f for f in os.listdir(preprocessed_patches_dir_luad) if category_label in f]
    for filename in filenames:
        patches = np.load(os.path.join(preprocessed_patches_dir_luad, filename))
        patches_list.append(patches)
    return np.concatenate(patches_list, axis=0) if patches_list else np.array([])

def load_patches_lusc(category_label):
    patches_list = []
    filenames = [f for f in os.listdir(preprocessed_patches_dir_lusc) if category_label in f]
    for filename in filenames:
        patches = np.load(os.path.join(preprocessed_patches_dir_lusc, filename))
        patches_list.append(patches)
    return np.concatenate(patches_list, axis=0) if patches_list else np.array([])

def load_patches_coad(category_label):
    patches_list = []
    filenames = [f for f in os.listdir(preprocessed_patches_dir_coad) if category_label in f]
    for filename in filenames:
        patches = np.load(os.path.join(preprocessed_patches_dir_coad, filename))
        patches_list.append(patches)
    return np.concatenate(patches_list, axis=0) if patches_list else np.array([])

In [6]:
brca_patches = load_patches_brca("BRCA")
luad_patches = load_patches_luad("LUAD")
lusc_patches = load_patches_lusc("LUSC")
coad_patches = load_patches_coad("COAD")

In [7]:
def embed(patches, model, preprocess, device, batch_size=64, verbose=True):
    num_batches = ceil(len(patches) / batch_size)
    opt_embs = []

    for batch_idx in tqdm(range(num_batches), disable=not verbose):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(patches))
        batch_np = patches[start:end]

        try:
            # Convert numpy arrays to PIL Images
            batch_pil = [Image.fromarray(patch.astype('uint8')) for patch in batch_np]
            
            # Apply CONCH preprocessing to each image individually
            batch_transformed = torch.stack([preprocess(img) for img in batch_pil])
            
            # Move batch to device
            batch = batch_transformed.to(device)

            # Get embeddings
            with torch.no_grad():
                batch_emb = model.encode_image(batch)

            # Move to CPU and append
            opt_embs.append(batch_emb.cpu())

        except Exception as e:
            print(f"Error processing batch {batch_idx}: {str(e)}")
            continue

    if not opt_embs:
        return np.array([])

    # Stack all embeddings
    opt_embs = torch.cat(opt_embs, dim=0)
    
    return opt_embs.numpy()

In [8]:
def embed_patches(patches, model, transform, device):
    if len(patches) == 0:
        return np.array([])
    model = model.to(device)
    model.eval()
    return embed(patches, model, transform, device)

In [9]:
brca_embeddings = embed_patches(brca_patches, model, preprocess, device)
num_brca = len(brca_embeddings)
brca_labels = [f"BRCA_{i+1}" for i in range(num_brca)]
np.save("brca_embeddings_conch.npy", brca_embeddings)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 977/977 [48:23<00:00,  2.97s/it]


In [10]:
luad_embeddings = embed_patches(luad_patches, model, preprocess, device)
num_luad = len(luad_embeddings)
luad_labels = [f"LUAD_{i+1}" for i in range(num_luad)]
np.save("luad_embeddings_conch.npy", luad_embeddings)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 977/977 [29:55<00:00,  1.84s/it]


In [11]:
lusc_embeddings = embed_patches(lusc_patches, model, preprocess, device)
num_lusc = len(lusc_embeddings)
lusc_labels = [f"LUSC_{i+1}" for i in range(num_lusc)]
np.save("lusc_embeddings_conch.npy", lusc_embeddings)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 977/977 [31:24<00:00,  1.93s/it]


In [12]:
coad_embeddings = embed_patches(coad_patches, model, preprocess, device)
num_coad = len(coad_embeddings)
coad_labels = [f"COAD_{i+1}" for i in range(num_coad)]
np.save("coad_embeddings_conch.npy", coad_embeddings)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 977/977 [28:54<00:00,  1.78s/it]
